# SPECTRA-ECG — META FINAL X10 MULTIMODAL V1.3 — META RESPONSE FIX

## Frozen V5.2 spectral routing → ECG visual panels → Meta Muse Spark 1.3 → structured evidence

**Locked hackathon experiment; non-diagnostic research prototype.**

This notebook upgrades the descriptor-only `SPECTRA_ECG_END_TO_END_V1_1_META_KEYFIX.ipynb` to a genuine multimodal pipeline.

**Hard constraints**

- routing comes from the published frozen V5/V5.2 Walsh12 scores; no router is retrained or retuned here;
- INCART exact lead II, non-overlapping 5-s windows;
- Muse sees waveform images + window IDs only: no endpoint labels, WFDB symbols, event counts, patient outcomes, or Walsh scores;
- ground truth is joined only after blind Meta outputs are written and SHA-256 sealed;
- primary routes: `FULL_META`, `SPECTRA_10_META`, `SPECTRA_20_META`;
- a pilot precedes the locked spend; `RUN_LOCKED=False` by default;
- optional uniform controls exist but are off by default;
- all route membership, prompts, usage, outputs, hashes and X10 checks are archived.

**Claim boundary:** this tests a systems-level quality–compute trade-off. It does not establish clinical diagnosis or Walsh-specific superiority. V5.2 already showed HaarDWT higher pooled event-weighted coverage than Walsh12 at the tested budgets, with patient-balanced Walsh–Haar differences inconclusive.

Frozen archive: DOI `10.5281/zenodo.23866944`.

V5.2 integrity anchors: 44 INCART records, 20 non-legacy patients, 15,840 windows, 17,830 endpoint events; Walsh12 pooled coverage ≈14.05% at 10% budget and ≈26.28% at 20%.
**V1.1 DUPFIX:** archive-table discovery now prioritizes the exact 15,840-window frozen table; long-form duplicates are accepted only after explicit Walsh12 filtering or proof that all duplicate rows are identical in score, endpoint count, and patient ID.

**V1.2 correction:** the two 15,840-row frozen tables are now assigned by exact semantic filename. `V5_BLIND_FEATURES_AND_SCORES_LOCKED.csv` is the only pre-inference routing source. `V5_PRIMARY_WITH_GROUND_TRUTH.csv` is not read until blind Meta outputs are SHA-256 sealed.

**V1.3 Meta-response fix:** Muse Spark is now called with `reasoning_effort='minimal'` and `max_completion_tokens=6000`; temperature is left unset. A `content=None` response is captured with finish reason, refusal, usage, reasoning-token diagnostics and the raw SDK response, and is never automatically retried.


## 0. Run order

1. Install/import.
2. Download or upload the DOI-stamped V5.2 ZIP.
3. Discover and verify the frozen score/truth table.
4. Lock a label-independent Meta subset by SHA-256 ordering of record IDs.
5. Freeze route membership from archived Walsh12 scores.
6. Download only required INCART `.hea/.dat` files and render ECG panels.
7. Connect to Meta Model API.
8. Run exactly one pilot panel per primary route.
9. Inspect JSON validity and projected cost.
10. Set `RUN_LOCKED=True` only after accepting the pilot.
11. Run locked inference.
12. Join truth only after the inference seal.
13. FINAL-X10 + export.

The notebook fails closed if V5.2 discovery/integrity fails; it never silently reconstructs or retunes the scout.

In [ ]:
!pip -q install "wfdb>=4.1" "openai>=1.50" pandas numpy matplotlib requests scipy


In [ ]:
import os,re,json,time,base64,hashlib,shutil,zipfile,warnings,math,getpass
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests, wfdb
warnings.filterwarnings("ignore")
SEED=42; np.random.seed(SEED)

ROOT=Path("/content/SPECTRA_META_FINAL_X10")
ARCHIVE_DIR=ROOT/"v5_2_archive"; INC_DIR=ROOT/"incartdb"; PANELS_DIR=ROOT/"panels"
OUT=ROOT/"outputs"; CACHE=ROOT/"api_cache"
for p in [ROOT,ARCHIVE_DIR,INC_DIR,PANELS_DIR,OUT,CACHE]: p.mkdir(parents=True,exist_ok=True)

MODEL_ID="muse-spark-1.3"; API_BASE="https://api.meta.ai/v1"
INPUT_USD_PER_1M=1.25; CACHED_INPUT_USD_PER_1M=0.15; OUTPUT_USD_PER_1M=4.25
WINDOW_SEC=5.0; PANEL_WINDOWS=8
N_META_RECORDS=int(os.getenv("N_META_RECORDS","4"))

# Safety gates
RUN_LOCKED=False
RUN_UNIFORM_CONTROLS=False

ZENODO_RECORD="23866944"; ZENODO_DOI="10.5281/zenodo.23866944"
ZENODO_ZIP_NAME="SPECTRA_ECG_V5_2_ZENODO_23866944.zip"
ZENODO_DIRECT=f"https://zenodo.org/records/{ZENODO_RECORD}/files/{ZENODO_ZIP_NAME}?download=1"

EXPECTED={"records":44,"patients":20,"windows":15840,"events":17830,
          "coverage_10":0.14049,"coverage_20":0.26276}

def sha256_file(path,block=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(block),b""): h.update(b)
    return h.hexdigest()

print("ROOT:",ROOT); print("Model:",MODEL_ID); print("RUN_LOCKED:",RUN_LOCKED)


## 1. Obtain and extract the frozen V5.2 archive

Automatic Zenodo download is attempted first. If Colab cannot reach Zenodo, upload the already-published DOI-stamped ZIP when prompted.

In [ ]:
zip_path=ROOT/ZENODO_ZIP_NAME
if not zip_path.exists():
    try:
        print("Downloading V5.2 from Zenodo...")
        with requests.get(ZENODO_DIRECT,stream=True,timeout=120) as r:
            r.raise_for_status()
            with open(zip_path,"wb") as f:
                for chunk in r.iter_content(1024*1024):
                    if chunk: f.write(chunk)
    except Exception as e:
        print("Automatic download failed:",type(e).__name__,str(e)[:400])
        try:
            from google.colab import files
            uploaded=files.upload()
            if ZENODO_ZIP_NAME not in uploaded: raise RuntimeError("Expected DOI-stamped ZIP.")
            open(zip_path,"wb").write(uploaded[ZENODO_ZIP_NAME])
        except Exception as e2:
            raise RuntimeError("V5.2 archive unavailable. Upload it; do not reconstruct the scout.") from e2

print("ZIP MB:",round(zip_path.stat().st_size/1024**2,2))
print("ZIP SHA-256:",sha256_file(zip_path))

if not any(ARCHIVE_DIR.iterdir()):
    with zipfile.ZipFile(zip_path) as z: z.extractall(ARCHIVE_DIR)

# Recursively unpack nested ZIPs because the DOI package may wrap the original frozen V5 release.
seen=set(); changed=True
while changed:
    changed=False
    for zp in list(ARCHIVE_DIR.rglob("*.zip")):
        key=str(zp.resolve())
        if key in seen: continue
        seen.add(key); target=zp.with_suffix(""); target.mkdir(exist_ok=True)
        try:
            with zipfile.ZipFile(zp) as z: z.extractall(target)
            changed=True
        except zipfile.BadZipFile: pass

all_files=[p for p in ARCHIVE_DIR.rglob("*") if p.is_file()]
print("Extracted files:",len(all_files))
for p in sorted(all_files)[:80]: print(p.relative_to(ARCHIVE_DIR))


## 2. Discover the frozen V5 score/truth table by content

The package is searched for a large CSV containing record ID, window ID/index, Walsh12 score and endpoint event count. Ambiguous non-identical top candidates cause a hard stop.

In [ ]:
# V1.2: no heuristic table discovery. The frozen V5.2 package contains two
# semantically different 15,840-row tables and they MUST NOT be conflated:
#   1) BLIND_FEATURES_AND_SCORES: allowed before Meta inference.
#   2) PRIMARY_WITH_GROUND_TRUTH: forbidden until after blind inference is sealed.

blind_matches=list(ARCHIVE_DIR.rglob("V5_BLIND_FEATURES_AND_SCORES_LOCKED.csv"))
truth_matches=list(ARCHIVE_DIR.rglob("V5_PRIMARY_WITH_GROUND_TRUTH.csv"))

if len(blind_matches)!=1:
    raise RuntimeError(f"Expected exactly one frozen blind table, found {len(blind_matches)}: {blind_matches}")
if len(truth_matches)!=1:
    raise RuntimeError(f"Expected exactly one frozen truth table, found {len(truth_matches)}: {truth_matches}")

blind_table_path=blind_matches[0]
truth_table_path=truth_matches[0]   # path only; DO NOT READ before inference seal.

raw_blind=pd.read_csv(blind_table_path)

def require_col(df, name):
    if name not in df.columns:
        raise RuntimeError(f"Required column {name!r} absent. Columns: {list(df.columns)}")
    return name

REC_COL=require_col(raw_blind,"record")
WIN_COL=require_col(raw_blind,"window")
SCORE_COL=require_col(raw_blind,"score_Walsh12")
PATIENT_COL=require_col(raw_blind,"patient")

print("Frozen BLIND table:",blind_table_path.relative_to(ARCHIVE_DIR),raw_blind.shape)
print("Frozen TRUTH table path (SEALED; not read):",truth_table_path.relative_to(ARCHIVE_DIR))
print("Blind columns:",list(raw_blind.columns))


In [ ]:
def canonical_record(x):
    s=str(x).strip()
    m=re.fullmatch(r"I?(\d{1,2})",s,re.I)
    if m: return f"I{int(m.group(1)):02d}"
    m=re.search(r"(?:^|[/_\-])I?(\d{1,2})$",s,re.I)
    return f"I{int(m.group(1)):02d}" if m else s

def canonical_window(s):
    x=pd.to_numeric(s,errors="coerce")
    if x.notna().all(): return x.astype(int)
    vals=[]
    for v in s.astype(str):
        m=re.search(r"[Ww](\d+)$",v)
        if m: vals.append(int(m.group(1))); continue
        nums=re.findall(r"(\d+)",v)
        vals.append(int(nums[-1]) if nums else None)
    out=pd.Series(vals,index=s.index,dtype="Int64")
    if out.isna().any(): raise RuntimeError("Cannot canonicalize all window IDs.")
    return out.astype(int)

blind=pd.DataFrame({
    "record":raw_blind[REC_COL].map(canonical_record),
    "window":canonical_window(raw_blind[WIN_COL]),
    "patient_id":raw_blind[PATIENT_COL].astype(str),
    "walsh_score":pd.to_numeric(raw_blind[SCORE_COL],errors="coerce")
})

if blind.walsh_score.isna().any(): raise RuntimeError("NaN/non-numeric frozen Walsh12 score.")
if blind[["record","window"]].duplicated().any():
    d=blind[blind.duplicated(["record","window"],keep=False)].sort_values(["record","window"])
    d.to_csv(OUT/"BLIND_DUPLICATE_DIAGNOSTIC.csv",index=False)
    raise RuntimeError("Frozen blind table itself has duplicate record/window keys; stop.")

blind["window_id"]=[f"{r}_W{int(w):04d}" for r,w in zip(blind.record,blind.window)]
blind["start_sec"]=blind.window*WINDOW_SEC
blind["end_sec"]=blind.start_sec+WINDOW_SEC

print("BLIND ONLY — no endpoint truth has been read.")
print("records",blind.record.nunique(),
      "patients",blind.patient_id.nunique(),
      "windows",len(blind))
display(blind.head())


## 3. V5.2 fail-closed integrity anchor

The inherited recordwise top-k rule is `ceil(rho*n_record)`, with stable score-descending/window-ascending ordering. Recomputed 10%/20% coverage must match the published V5.2 values to rounding tolerance before any Meta call.

In [ ]:
# PRE-INFERENCE integrity checks are deliberately truth-free.
pre_integrity={
    "blind_records_44":blind.record.nunique()==EXPECTED["records"],
    "blind_patients_20":blind.patient_id.nunique()==EXPECTED["patients"],
    "blind_windows_15840":len(blind)==EXPECTED["windows"],
    "blind_unique_record_window":not blind[["record","window"]].duplicated().any(),
    "blind_scores_finite":np.isfinite(blind.walsh_score.to_numpy()).all()
}
pre_integrity_df=pd.DataFrame([{"check":k,"passed":bool(v)} for k,v in pre_integrity.items()])
display(pre_integrity_df)
if not pre_integrity_df.passed.all():
    raise RuntimeError("Frozen blind V5 table failed pre-inference integrity checks.")

# No truth object exists at this stage by design.
blind.to_csv(OUT/"FROZEN_BLIND_ROUTING_TABLE.csv",index=False)

# Fingerprint the exact blind input used for routing.
BLIND_TABLE_SHA256=sha256_file(blind_table_path)
print("PRE-INFERENCE BLIND integrity PASS.")
print("Blind table SHA-256:",BLIND_TABLE_SHA256)
print("Ground truth remains SEALED on disk and has not been loaded.")


## 4. Lock a label-independent Meta subset and route membership

Default: four V5.2 records selected by SHA-256 ordering of record IDs only. This limits hackathon API cost without selecting on endpoint labels or scout scores.

`FULL_META` = all windows in those records.  
`SPECTRA_10_META` / `SPECTRA_20_META` = frozen Walsh12 recordwise top-k.

The Walsh score itself is never sent to Muse.

In [ ]:
def subset_key(r): return hashlib.sha256(("SPECTRA-META-FINAL-X10|"+str(r)).encode()).hexdigest()
all_records=sorted(blind.record.unique())
META_RECORDS=sorted(all_records,key=subset_key)[:N_META_RECORDS]
meta_blind=blind[blind.record.isin(META_RECORDS)].copy().reset_index(drop=True)

selection_payload={"rule":"SHA256('SPECTRA-META-FINAL-X10|'+record)","N":N_META_RECORDS,"records":META_RECORDS}
SELECTION_SHA256=hashlib.sha256(json.dumps(selection_payload,sort_keys=True,separators=(",",":")).encode()).hexdigest()
json.dump(selection_payload,open(OUT/"locked_meta_subset.json","w"),indent=2)

def topk_ids(df,rho):
    ids=[]
    for rec,g in df.groupby("record",sort=True):
        k=max(1,int(np.ceil(rho*len(g))))
        q=g.sort_values(["walsh_score","window"],ascending=[False,True],kind="mergesort")
        ids+=q.head(k).window_id.tolist()
    return ids

ROUTES={"FULL_META":meta_blind.window_id.tolist(),
        "SPECTRA_10_META":topk_ids(meta_blind,.10),
        "SPECTRA_20_META":topk_ids(meta_blind,.20)}

if RUN_UNIFORM_CONTROLS:
    rng=np.random.default_rng(SEED)
    for rho in [.10,.20]:
        ids=[]
        for rec,g in meta_blind.groupby("record",sort=True):
            k=max(1,int(np.ceil(rho*len(g))))
            ids+=rng.choice(g.window_id.to_numpy(),size=k,replace=False).tolist()
        ROUTES[f"UNIFORM_{int(rho*100)}_META"]=ids

route_membership=pd.DataFrame([{"route":r,"window_id":w} for r,ids in ROUTES.items() for w in ids])
route_membership.to_csv(OUT/"LOCKED_ROUTE_MEMBERSHIP.csv",index=False)
route_hash=hashlib.sha256(route_membership.sort_values(["route","window_id"]).to_csv(index=False).encode()).hexdigest()

print("META_RECORDS:",META_RECORDS)
print("subset windows:",len(meta_blind),"selection SHA:",SELECTION_SHA256)
for r,ids in ROUTES.items(): print(r,len(ids),f"{len(ids)/len(meta_blind):.2%}")
print("route SHA:",route_hash)


## 5. Download waveform-only INCART data and render exact lead-II panels

Only `.hea` and `.dat` are read before inference. WFDB annotations are not loaded. Each 5-s waveform is robust-centered/scaled **for visualization only**; the archived scout score remains untouched.

In [ ]:
for i,rec in enumerate(META_RECORDS,1):
    need=[f"{rec}.dat",f"{rec}.hea"]
    if not all((INC_DIR/f).exists() for f in need):
        print("INCART",i,"/",len(META_RECORDS),rec)
        wfdb.dl_files("incartdb",str(INC_DIR),need)

def lead_ii_index(rec):
    h=wfdb.rdheader(str(INC_DIR/rec)); names=[str(x).strip() for x in h.sig_name]
    ids=[i for i,n in enumerate(names) if n.upper()=="II"]
    if len(ids)!=1: raise RuntimeError(f"{rec}: exact lead II not unique: {names}")
    return ids[0]

signals={}
for rec in META_RECORDS:
    rr=wfdb.rdrecord(str(INC_DIR/rec)); ch=lead_ii_index(rec)
    signals[rec]={"x":rr.p_signal[:,ch].astype(float),"fs":float(rr.fs),"channel":ch}
    print(rec,"fs",rr.fs,"lead-II channel",ch)


In [ ]:
def rz_visual(x):
    x=np.asarray(x,float); med=np.median(x); mad=np.median(np.abs(x-med)); s=1.4826*mad
    if not np.isfinite(s) or s<1e-8: s=np.std(x)+1e-8
    return (x-med)/s

def waveform_for_window(wid):
    row=meta_blind.loc[meta_blind.window_id.eq(wid)].iloc[0]
    rec=row.record; wi=int(row.window); x=signals[rec]["x"]; fs=signals[rec]["fs"]
    wn=int(round(WINDOW_SEC*fs)); st=wi*wn; en=st+wn
    if en>len(x): raise RuntimeError(f"{wid}: exceeds waveform.")
    return x[st:en],fs,row

def render_panel(ids,path):
    n=len(ids); nc=2; nr=int(np.ceil(n/nc))
    fig,axs=plt.subplots(nr,nc,figsize=(14,2.6*nr),squeeze=False); axs=axs.ravel()
    for ax,wid in zip(axs,ids):
        x,fs,row=waveform_for_window(wid); z=rz_visual(x); t=np.arange(len(z))/fs
        ax.plot(t,z,linewidth=.8); ax.set_xlim(0,WINDOW_SEC); ax.set_title(wid,fontsize=10)
        ax.set_xlabel("time (s)",fontsize=8); ax.set_ylabel("robust-normalized amplitude",fontsize=8)
        ax.grid(alpha=.18)
    for ax in axs[n:]: ax.axis("off")
    fig.suptitle("ECG lead II — 5-s windows — non-diagnostic review evidence",fontsize=13)
    fig.tight_layout(); fig.savefig(path,dpi=150,bbox_inches="tight"); plt.close(fig)
    return path

smoke_ids=ROUTES["SPECTRA_10_META"][:min(PANEL_WINDOWS,len(ROUTES["SPECTRA_10_META"]))]
smoke=PANELS_DIR/"SMOKE_PANEL.png"; render_panel(smoke_ids,smoke)
print(smoke); display(plt.imread(smoke))


## 6. Secure Meta Model API connection

Use the existing `SPECTRA-ECG-Colab` key as Colab secret `MODEL_API_KEY`. The key is never printed or exported.

If the Meta dashboard billing restriction is still active, this cell should fail before any experiment spend.

In [ ]:
from openai import OpenAI
API_KEY=os.getenv("MODEL_API_KEY","").strip(); source="environment"
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY=(userdata.get("MODEL_API_KEY") or "").strip()
        if API_KEY: source="Colab Secret"
    except Exception: pass
if not API_KEY:
    API_KEY=getpass.getpass("Meta MODEL_API_KEY (hidden): ").strip(); source="hidden prompt"
if not API_KEY: raise RuntimeError("No MODEL_API_KEY.")
client=OpenAI(base_url=API_BASE,api_key=API_KEY)
try:
    ping=client.chat.completions.create(model=MODEL_ID,
        messages=[{"role":"user","content":"Reply with the single word: ready"}],max_tokens=16)
    print("Meta connectivity:",ping.choices[0].message.content)
    print("Key source:",source,"(secret not displayed)")
except Exception as e:
    raise RuntimeError("Meta connectivity failed; resolve account/billing restriction and rerun.") from e


## 7. Locked multimodal prompt + strict structured output

Muse receives one ECG panel and the window IDs visible in it. Route name, Walsh score and all endpoint truth are withheld.

In [ ]:
SYSTEM_PROMPT = """You are a non-diagnostic ECG review-prioritization component.
You inspect images containing multiple 5-second lead-II ECG windows.
For each labeled window, assess only whether its visible waveform deserves priority human review
because of potentially unusual rhythm/morphology or signal-quality concerns.
Do not diagnose disease, do not name a patient condition, and do not infer hidden labels.
Use only what is visibly present in the supplied ECG panel.
Return exactly one structured result for every supplied window ID."""
PROMPT_VERSION="META_FINAL_X10_MULTIMODAL_V1_3"

def schema(ids):
    return {"type":"json_schema","json_schema":{"name":"ecg_review_panel","strict":True,"schema":{
        "type":"object","properties":{"windows":{"type":"array","items":{"type":"object",
        "properties":{
            "window_id":{"type":"string","enum":list(ids)},
            "review":{"type":"boolean"},
            "priority":{"type":"string","enum":["low","medium","high"]},
            "signal_quality":{"type":"string","enum":["adequate","limited","poor"]},
            "confidence":{"type":"number"},
            "visual_evidence":{"type":"string"}},
        "required":["window_id","review","priority","signal_quality","confidence","visual_evidence"],
        "additionalProperties":False}}},
        "required":["windows"],"additionalProperties":False}}}

def data_url(path):
    return "data:image/png;base64,"+base64.b64encode(open(path,"rb").read()).decode("ascii")

def usage(resp):
    u=getattr(resp,"usage",None)
    if u is None: return dict(input_tokens=np.nan,cached_input_tokens=np.nan,output_tokens=np.nan,total_tokens=np.nan)
    inp=getattr(u,"prompt_tokens",getattr(u,"input_tokens",np.nan))
    out=getattr(u,"completion_tokens",getattr(u,"output_tokens",np.nan))
    tot=getattr(u,"total_tokens",np.nan); cached=np.nan
    det=getattr(u,"prompt_tokens_details",None)
    if det is not None: cached=getattr(det,"cached_tokens",np.nan)
    return dict(input_tokens=inp,cached_input_tokens=cached,output_tokens=out,total_tokens=tot)

def call_panel(ids,panel_path,stage):
    ids=list(ids)
    digest=sha256_file(panel_path)
    ck=hashlib.sha256(
        (PROMPT_VERSION+"|"+MODEL_ID+"|"+digest+"|"+"|".join(ids)).encode()
    ).hexdigest()
    cf=CACHE/(ck+".json")
    if cf.exists():
        z=json.load(open(cf))
        z["cached_local"]=True
        return z

    text=(
        "Analyze the ECG panel. The labels correspond exactly to these window IDs: "
        + json.dumps(ids)
        + ". Return one result per ID. Research review prioritization only; no diagnosis."
    )
    content=[
        {"type":"text","text":text},
        {"type":"image_url","image_url":{"url":data_url(panel_path),"detail":"high"}}
    ]

    t0=time.perf_counter()
    resp=client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role":"developer","content":SYSTEM_PROMPT},
            {"role":"user","content":content}
        ],
        response_format=schema(ids),
        # Muse Spark is a reasoning model. Reasoning tokens count against the
        # completion budget. This is a classification/extraction task, so use
        # the minimum supported reasoning effort and leave temperature unset.
        reasoning_effort="minimal",
        max_completion_tokens=6000
    )
    latency=time.perf_counter()-t0

    choice=resp.choices[0]
    msg=choice.message
    raw_content=getattr(msg,"content",None)
    finish_reason=getattr(choice,"finish_reason",None)
    refusal=getattr(msg,"refusal",None)
    u=usage(resp)

    # Always preserve a diagnostic snapshot BEFORE parsing.
    diagnostic={
        "stage":stage,
        "model":MODEL_ID,
        "prompt_version":PROMPT_VERSION,
        "panel_sha256":digest,
        "window_ids":ids,
        "finish_reason":finish_reason,
        "content_is_none":raw_content is None,
        "refusal":refusal,
        "latency_sec":latency,
        **{k:(None if pd.isna(v) else int(v)) for k,v in u.items()}
    }

    # Capture reasoning-token count when exposed by the SDK.
    try:
        det=getattr(resp.usage,"completion_tokens_details",None)
        diagnostic["reasoning_tokens"]=(
            None if det is None else getattr(det,"reasoning_tokens",None)
        )
    except Exception:
        diagnostic["reasoning_tokens"]=None

    diag_path=OUT/f"META_RESPONSE_DIAGNOSTIC_{stage}_{ck[:12]}.json"
    json.dump(diagnostic,open(diag_path,"w"),indent=2,default=str)

    if raw_content is None:
        # Preserve the full SDK response locally when possible. Do NOT
        # automatically retry: an automatic retry could spend credits while
        # hiding the actual failure mode.
        try:
            full=resp.model_dump(mode="json")
            json.dump(full,open(OUT/f"META_RAW_RESPONSE_{stage}_{ck[:12]}.json","w"),
                      indent=2,default=str)
        except Exception:
            pass
        raise RuntimeError(
            "Meta returned message.content=None. "
            f"finish_reason={finish_reason!r}, refusal={refusal!r}, "
            f"usage={u}, diagnostic={diag_path}. "
            "No automatic retry was made."
        )

    try:
        parsed=json.loads(raw_content)
    except Exception as e:
        open(OUT/f"META_UNPARSED_CONTENT_{stage}_{ck[:12]}.txt",
             "w",encoding="utf-8").write(str(raw_content))
        raise RuntimeError(
            f"Meta returned non-parseable structured content; finish_reason={finish_reason!r}. "
            f"Saved diagnostic to {diag_path}"
        ) from e

    arr=parsed.get("windows")
    if not isinstance(arr,list):
        raise RuntimeError(f"Structured response lacks windows[]; diagnostic={diag_path}")

    got=[str(x["window_id"]) for x in arr]
    if len(arr)!=len(ids) or set(got)!=set(ids) or len(got)!=len(set(got)):
        raise RuntimeError(
            f"Window-ID/schema mismatch: expected {ids}, got {got}; diagnostic={diag_path}"
        )

    z={
        "stage":stage,
        "model":MODEL_ID,
        "prompt_version":PROMPT_VERSION,
        "panel_sha256":digest,
        "window_ids":ids,
        "parsed":parsed,
        "latency_sec":latency,
        "finish_reason":finish_reason,
        "reasoning_effort":"minimal",
        "max_completion_tokens":6000,
        **{k:(None if pd.isna(v) else int(v)) for k,v in u.items()},
        "reasoning_tokens":diagnostic.get("reasoning_tokens"),
        "cached_local":False
    }
    json.dump(z,open(cf,"w"),indent=2,default=str)
    return z


## 8. Pilot — one panel per primary route

Pilot checks image transport, strict JSON, ID preservation, token usage, latency and projected cost. Endpoint truth remains sealed.

If a semantic prompt change is made after the pilot, increment `PROMPT_VERSION` and repeat the pilot.

In [ ]:
pilot_calls=[]; pilot_decisions=[]
for route in ["FULL_META","SPECTRA_10_META","SPECTRA_20_META"]:
    ids=ROUTES[route][:min(PANEL_WINDOWS,len(ROUTES[route]))]
    p=PANELS_DIR/f"PILOT_{route}.png"; render_panel(ids,p); z=call_panel(ids,p,"pilot")
    pilot_calls.append({"route":route,"n_windows":len(ids),"latency_sec":z["latency_sec"],
        "input_tokens":z["input_tokens"],"cached_input_tokens":z["cached_input_tokens"],
        "output_tokens":z["output_tokens"],"total_tokens":z["total_tokens"],
        "cached_local":z["cached_local"]})
    for x in z["parsed"]["windows"]: pilot_decisions.append({"route":route,**x})

pilot_calls=pd.DataFrame(pilot_calls); pilot_decisions=pd.DataFrame(pilot_decisions)
pilot_calls.to_csv(OUT/"pilot_api_calls.csv",index=False); pilot_decisions.to_csv(OUT/"pilot_meta_outputs.csv",index=False)
display(pilot_calls); display(pilot_decisions)


In [ ]:
def estimate_cost(row):
    inp=float(row.get("input_tokens",np.nan)); out=float(row.get("output_tokens",np.nan))
    cached=row.get("cached_input_tokens",0)
    cached=0 if pd.isna(cached) else float(cached)
    if not np.isfinite(inp) or not np.isfinite(out): return np.nan
    cached=min(max(cached,0),inp); uncached=inp-cached
    return uncached/1e6*INPUT_USD_PER_1M+cached/1e6*CACHED_INPUT_USD_PER_1M+out/1e6*OUTPUT_USD_PER_1M

pilot_calls["estimated_cost_usd"]=pilot_calls.apply(estimate_cost,axis=1)
mean_cost=pilot_calls.estimated_cost_usd.mean()
planned_panels=sum(math.ceil(len(ids)/PANEL_WINDOWS) for ids in ROUTES.values())
projected=mean_cost*planned_panels if np.isfinite(mean_cost) else np.nan
pilot_calls.to_csv(OUT/"pilot_cost_projection.csv",index=False)
print("Mean pilot cost/panel:",round(mean_cost,6) if np.isfinite(mean_cost) else "NA")
print("Planned locked panels:",planned_panels)
print("Projected locked cost:",round(projected,3) if np.isfinite(projected) else "NA")
print("RUN_LOCKED:",RUN_LOCKED)
if not RUN_LOCKED:
    print("SAFETY STOP: inspect pilot, then set RUN_LOCKED=True in configuration before the locked run.")


## 9. Locked multimodal run — spending cell

This cell does nothing while `RUN_LOCKED=False`. Completed panels are locally cached by image hash + model + prompt version + IDs.

In [ ]:
if not RUN_LOCKED:
    print("LOCKED RUN NOT STARTED.")
else:
    api_rows=[]; decision_rows=[]
    for route,ids in ROUTES.items():
        print("\nROUTE",route,"windows",len(ids))
        panels=[ids[i:i+PANEL_WINDOWS] for i in range(0,len(ids),PANEL_WINDOWS)]
        for pi,pids in enumerate(panels):
            p=PANELS_DIR/f"{route}_P{pi:04d}.png"
            if not p.exists(): render_panel(pids,p)
            z=call_panel(pids,p,"locked")
            api_rows.append({"route":route,"panel":pi,"n_windows":len(pids),"panel_sha256":z["panel_sha256"],
                "latency_sec":z["latency_sec"],"input_tokens":z["input_tokens"],
                "cached_input_tokens":z["cached_input_tokens"],"output_tokens":z["output_tokens"],
                "total_tokens":z["total_tokens"],"cached_local":z["cached_local"]})
            for x in z["parsed"]["windows"]: decision_rows.append({"route":route,"panel":pi,**x})
            if (pi+1)%10==0 or pi+1==len(panels): print(f"  {pi+1}/{len(panels)} panels")

    locked_calls=pd.DataFrame(api_rows); locked_decisions=pd.DataFrame(decision_rows)
    locked_calls.to_csv(OUT/"LOCKED_API_CALLS.csv",index=False)
    locked_decisions.to_csv(OUT/"LOCKED_META_DECISIONS_BLIND.csv",index=False)
    seal={"decisions_sha256":sha256_file(OUT/"LOCKED_META_DECISIONS_BLIND.csv"),
          "api_calls_sha256":sha256_file(OUT/"LOCKED_API_CALLS.csv"),
          "route_membership_sha256":route_hash,"selection_sha256":SELECTION_SHA256,
          "prompt_version":PROMPT_VERSION,"model":MODEL_ID}
    json.dump(seal,open(OUT/"PRE_TRUTH_INFERENCE_SEAL.json","w"),indent=2)
    print("Blind inference sealed BEFORE truth join.")
    print("Decision SHA:",seal["decisions_sha256"])


## 10. Post-inference evaluation — truth enters only here

Report separately:

- **scout endpoint coverage** = events in all routed windows / all events;
- **end-to-end endpoint recall** = events in Muse-flagged routed windows / all events.

These are different estimands and must not be conflated.

In [ ]:
seal_path=OUT/"PRE_TRUTH_INFERENCE_SEAL.json"
decisions_path=OUT/"LOCKED_META_DECISIONS_BLIND.csv"
calls_path=OUT/"LOCKED_API_CALLS.csv"

truth_integrity={}

if not seal_path.exists():
    print("No locked inference seal; ground truth remains unread and evaluation is skipped.")
else:
    seal=json.load(open(seal_path))
    if sha256_file(decisions_path)!=seal["decisions_sha256"]:
        raise RuntimeError("Decision seal mismatch.")
    if sha256_file(calls_path)!=seal["api_calls_sha256"]:
        raise RuntimeError("Call-log seal mismatch.")
    if seal["route_membership_sha256"]!=route_hash or seal["selection_sha256"]!=SELECTION_SHA256:
        raise RuntimeError("Route/subset seal mismatch.")

    # ---- FIRST GROUND-TRUTH READ IN THE NOTEBOOK ----
    raw_truth=pd.read_csv(truth_table_path)
    for col in ["record","window","patient","score_Walsh12","events"]:
        if col not in raw_truth.columns:
            raise RuntimeError(f"Truth table missing {col!r}; columns={list(raw_truth.columns)}")

    truth=pd.DataFrame({
        "record":raw_truth["record"].map(canonical_record),
        "window":canonical_window(raw_truth["window"]),
        "patient_id":raw_truth["patient"].astype(str),
        "walsh_score_truthcopy":pd.to_numeric(raw_truth["score_Walsh12"],errors="coerce"),
        "endpoint_events":pd.to_numeric(raw_truth["events"],errors="coerce")
    })
    truth["window_id"]=[f"{r}_W{int(w):04d}" for r,w in zip(truth.record,truth.window)]
    truth["positive"]=(truth.endpoint_events>0).astype(int)

    if truth[["record","window"]].duplicated().any():
        raise RuntimeError("Truth table has duplicate record/window keys.")
    if truth[["walsh_score_truthcopy","endpoint_events"]].isna().any().any():
        raise RuntimeError("NaN/non-numeric truth-table score/events.")

    # Verify blind and truth tables are the same frozen window universe and scores.
    cross=blind[["window_id","walsh_score"]].merge(
        truth[["window_id","walsh_score_truthcopy"]],
        on="window_id",how="outer",indicator=True,validate="one_to_one")
    max_score_diff=np.nanmax(np.abs(cross.walsh_score-cross.walsh_score_truthcopy))
    truth_integrity={
        "truth_records_44":truth.record.nunique()==EXPECTED["records"],
        "truth_patients_20":truth.patient_id.nunique()==EXPECTED["patients"],
        "truth_windows_15840":len(truth)==EXPECTED["windows"],
        "truth_events_17830":int(truth.endpoint_events.sum())==EXPECTED["events"],
        "blind_truth_same_window_universe":(cross["_merge"]=="both").all(),
        "blind_truth_scores_identical":np.isfinite(max_score_diff) and max_score_diff<1e-12
    }

    # Recompute published V5.2 Walsh coverage only now, after inference was sealed.
    def truth_topk_coverage(rho):
        ids=set()
        for rec,g in truth.groupby("record",sort=True):
            k=max(1,int(np.ceil(rho*len(g))))
            q=g.sort_values(["walsh_score_truthcopy","window"],ascending=[False,True],kind="mergesort")
            ids.update(q.head(k).window_id)
        return truth.loc[truth.window_id.isin(ids),"endpoint_events"].sum()/truth.endpoint_events.sum()

    cov10=truth_topk_coverage(.10); cov20=truth_topk_coverage(.20)
    truth_integrity["coverage_10_matches_V5"]=abs(cov10-EXPECTED["coverage_10"])<0.00075
    truth_integrity["coverage_20_matches_V5"]=abs(cov20-EXPECTED["coverage_20"])<0.00075

    truth_integrity_df=pd.DataFrame([{"check":k,"passed":bool(v)} for k,v in truth_integrity.items()])
    display(truth_integrity_df)
    print(f"Post-seal V5.2 coverage anchors: 10%={cov10:.6f}, 20%={cov20:.6f}")
    if not truth_integrity_df.passed.all():
        raise RuntimeError("Post-seal V5.2 truth integrity failed; do not interpret Meta results.")

    # Only now restrict truth to the pre-locked Meta records.
    d=pd.read_csv(decisions_path)
    calls=pd.read_csv(calls_path)
    meta_truth=truth[truth.record.isin(META_RECORDS)].copy()
    total_events=float(meta_truth.endpoint_events.sum())
    total_windows=len(meta_truth)

    joined=d.merge(
        meta_truth[["window_id","record","window","patient_id","endpoint_events","positive"]],
        on="window_id",how="left",validate="many_to_one")
    if joined.endpoint_events.isna().any():
        raise RuntimeError("Missing ground truth after post-seal join.")

    rt=route_membership.merge(
        meta_truth[["window_id","endpoint_events","positive"]],
        on="window_id",how="left",validate="many_to_one")

    quality=[]
    for route,ids in ROUTES.items():
        a=rt[rt.route.eq(route)]
        q=joined[joined.route.eq(route)].copy()
        flag=q.review.astype(bool)
        scout_events=float(a.endpoint_events.sum())
        meta_events=float(q.loc[flag,"endpoint_events"].sum())
        fw=int(flag.sum())
        pw=int(q.loc[flag,"positive"].sum())
        quality.append({
            "route":route,
            "ecg_windows_total":total_windows,
            "windows_sent_to_meta":len(ids),
            "fraction_sent_to_meta":len(ids)/total_windows,
            "scout_endpoint_coverage":scout_events/total_events if total_events else np.nan,
            "meta_flagged_windows":fw,
            "fraction_total_ecg_flagged":fw/total_windows,
            "end_to_end_endpoint_recall":meta_events/total_events if total_events else np.nan,
            "positive_window_precision_among_meta_flagged":pw/fw if fw else np.nan,
            "meta_review_yield":fw/len(ids) if ids else np.nan
        })
    quality=pd.DataFrame(quality)

    calls["estimated_cost_usd"]=calls.apply(estimate_cost,axis=1)
    compute=calls.groupby("route",as_index=False).agg(
        api_calls=("panel","count"),
        input_tokens=("input_tokens","sum"),
        cached_input_tokens=("cached_input_tokens","sum"),
        output_tokens=("output_tokens","sum"),
        total_tokens=("total_tokens","sum"),
        api_latency_sum_sec=("latency_sec","sum"),
        api_latency_mean_sec=("latency_sec","mean"),
        estimated_cost_usd=("estimated_cost_usd","sum")
    )
    summary=quality.merge(compute,on="route",validate="one_to_one")
    full=summary.loc[summary.route.eq("FULL_META")].iloc[0]

    for col in ["windows_sent_to_meta","api_calls","input_tokens","output_tokens","total_tokens",
                "api_latency_sum_sec","estimated_cost_usd"]:
        base=float(full[col])
        summary["reduction_vs_full_"+col]=1-summary[col].astype(float)/base if base else np.nan

    fr=float(full.end_to_end_endpoint_recall)
    summary["fraction_of_full_end_to_end_recall_retained"]=(
        summary.end_to_end_endpoint_recall/fr if fr>0 else np.nan)

    summary.to_csv(OUT/"FINAL_ROUTE_SUMMARY.csv",index=False)
    joined.to_csv(OUT/"FINAL_WINDOW_LEVEL_EVALUATION.csv",index=False)
    truth_integrity_df.to_csv(OUT/"POST_SEAL_TRUTH_INTEGRITY.csv",index=False)
    display(summary)


## 11. Hackathon figures

In [ ]:
if (OUT/"FINAL_ROUTE_SUMMARY.csv").exists():
    summary=pd.read_csv(OUT/"FINAL_ROUTE_SUMMARY.csv")
    specs=[
      ("fraction_sent_to_meta","scout_endpoint_coverage","Fraction sent to Meta","Scout endpoint coverage",
       "Frozen routing quality–compute trade-off","fig_meta_scout_tradeoff.png"),
      ("fraction_sent_to_meta","end_to_end_endpoint_recall","Fraction sent to Meta","End-to-end endpoint recall",
       "SPECTRA → Muse Spark quality–compute trade-off","fig_meta_e2e_tradeoff.png"),
      ("estimated_cost_usd","end_to_end_endpoint_recall","Estimated Meta API cost (USD)","End-to-end endpoint recall",
       "Measured API cost vs retained evidence","fig_meta_cost_tradeoff.png")]
    for x,y,xl,yl,title,fn in specs:
        plt.figure(figsize=(7,5))
        for _,r in summary.iterrows():
            plt.scatter(r[x],r[y],s=70); plt.annotate(r.route,(r[x],r[y]),xytext=(5,5),textcoords="offset points")
        plt.xlabel(xl); plt.ylabel(yl); plt.title(title); plt.grid(alpha=.25); plt.tight_layout()
        plt.savefig(OUT/fn,dpi=180); plt.show()
else: print("Locked summary unavailable; figures skipped.")


## 12. FINAL-X10 adversarial audit

A locked result is interpretable only if all applicable checks pass.

In [ ]:
audit=[]
def add(name,v,note=""): audit.append({"check":name,"passed":bool(v),"note":note})

for k,v in pre_integrity.items(): add("pre_"+k,v)
add("standard_muse_spark_1_3",MODEL_ID=="muse-spark-1.3")
add("not_contributor","contributor" not in MODEL_ID.lower())
add("selection_hash",bool(SELECTION_SHA256))
add("route_hash",bool(route_hash))
add("full_route_all_subset",len(ROUTES["FULL_META"])==len(meta_blind))
add("spectra10_budget",.09<=len(ROUTES["SPECTRA_10_META"])/len(meta_blind)<=.12)
add("spectra20_budget",.19<=len(ROUTES["SPECTRA_20_META"])/len(meta_blind)<=.22)
add("truth_columns_absent_pre_inference",not {"endpoint_events","positive"}.intersection(meta_blind.columns))
add("no_walsh_score_in_system_prompt","walsh_score" not in SYSTEM_PROMPT.lower())
add("no_route_name_in_system_prompt","spectra_10_meta" not in SYSTEM_PROMPT.lower())

if seal_path.exists():
    seal=json.load(open(seal_path))
    add("decision_seal",sha256_file(decisions_path)==seal["decisions_sha256"])
    add("call_seal",sha256_file(calls_path)==seal["api_calls_sha256"])
    d=pd.read_csv(decisions_path)
    exp=route_membership[["route","window_id"]].sort_values(["route","window_id"]).reset_index(drop=True)
    got=d[["route","window_id"]].sort_values(["route","window_id"]).reset_index(drop=True)
    add("exact_one_decision_per_routed_window",exp.equals(got))
    add("no_duplicate_decisions",not d[["route","window_id"]].duplicated().any())
    add("primary_routes_complete",{"FULL_META","SPECTRA_10_META","SPECTRA_20_META"}.issubset(set(d.route)))
    if truth_integrity:
        for k,v in truth_integrity.items(): add("postseal_"+k,v)
else:
    add("locked_inference_seal_exists",False,"Expected false before RUN_LOCKED=True.")

if (OUT/"FINAL_ROUTE_SUMMARY.csv").exists():
    s=pd.read_csv(OUT/"FINAL_ROUTE_SUMMARY.csv")
    add("finite_primary_metrics",
        np.isfinite(s[["scout_endpoint_coverage","end_to_end_endpoint_recall"]].to_numpy()).all())
    add("full_scout_coverage_one",
        abs(float(s.loc[s.route.eq("FULL_META"),"scout_endpoint_coverage"].iloc[0])-1)<1e-12)

audit=pd.DataFrame(audit)
audit.to_csv(OUT/"FINAL_X10_AUDIT.csv",index=False)
display(audit)
if RUN_LOCKED and not audit.passed.all():
    raise RuntimeError("FINAL-X10 failed; do not interpret results.")
elif RUN_LOCKED:
    print("ALL APPLICABLE FINAL-X10 CHECKS PASSED.")
else:
    print("Pre-lock audit: missing locked seal is expected until RUN_LOCKED=True.")


## 13. Export reproducibility package

The ZIP contains route/subset locks, pilot logs, blind Muse outputs, post-inference metrics, usage/cost, figures, protocol metadata, X10 audit and SHA-256 manifest. It excludes the API key and raw PhysioNet waveforms.

In [ ]:
metadata={"experiment":"SPECTRA-ECG META FINAL X10 MULTIMODAL",
 "created_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
 "model_id":MODEL_ID,"api_base":API_BASE,"prompt_version":PROMPT_VERSION,
 "v5_2_zenodo_doi":ZENODO_DOI,"v5_2_archive_sha256":sha256_file(zip_path),
 "frozen_blind_table":str(blind_table_path.relative_to(ARCHIVE_DIR)),
 "sealed_truth_table":str(truth_table_path.relative_to(ARCHIVE_DIR)),
 "frozen_blind_table_sha256":BLIND_TABLE_SHA256,
 "selection_sha256":SELECTION_SHA256,"route_membership_sha256":route_hash,
 "meta_records":META_RECORDS,"window_seconds":WINDOW_SEC,"panel_windows":PANEL_WINDOWS,
 "routes":{k:len(v) for k,v in ROUTES.items()},"run_locked":RUN_LOCKED,
 "uniform_controls_enabled":RUN_UNIFORM_CONTROLS,
 "pricing":{"input":INPUT_USD_PER_1M,"cached_input":CACHED_INPUT_USD_PER_1M,"output":OUTPUT_USD_PER_1M,
            "units":"USD per 1M tokens; verify official pricing before any later rerun"},
 "scope":"non-diagnostic review prioritization research prototype",
 "truth_separation":"waveform images/window IDs only before sealed inference",
 "claim_boundary":"systems quality-compute trade-off; no Walsh superiority or clinical diagnosis"}
json.dump(metadata,open(OUT/"EXPERIMENT_METADATA.json","w"),indent=2)

manifest=[]
for p in sorted(OUT.rglob("*")):
    if p.is_file(): manifest.append({"file":str(p.relative_to(OUT)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
pd.DataFrame(manifest).to_csv(OUT/"SHA256_MANIFEST.csv",index=False)

base="/content/SPECTRA_ECG_META_FINAL_X10_MULTIMODAL_RESULTS"
shutil.make_archive(base,"zip",OUT)
print("Created",base+".zip")
print("SHA-256",sha256_file(base+".zip"))
print("MB",round(os.path.getsize(base+".zip")/1024**2,2))
try:
    from google.colab import files
    files.download(base+".zip")
except Exception: pass


# Interpretation contract

A successful locked run can support:

> On the pre-locked Meta evaluation subset, frozen spectral routing reduced the fraction of ECG windows sent to Muse Spark while retaining a measured fraction of the project-specific review-relevant endpoint and producing structured multimodal evidence reports.

It cannot support “Muse diagnoses arrhythmia”, “Walsh is superior to Haar”, or clinical validation. The full 44-record V5.2 coverage values must not be substituted for the smaller Meta subset; the notebook recomputes the subset values after the inference seal.